# 04 · Kümeleme karşılaştırması ve ortak ana model seçimi

URF + K-Medoids başlangıç referansıdır. Gower + K-Medoids ve uygulanabiliyorsa K-Prototypes
aynı alanlar ve dönemlerle karşılaştırılır. Alan oranları %25 / %50, banka toplam küme
sayıları 5 / 7’dir. URF mtry oranı **kodlama sonrası** sütun sayısının %25’idir.

İki coğrafi grubun ayrı fitleri vardır; seçilen yöntem ve senaryo bütün gruplar ve göstergeler
için ortaktır. Kümeleme gridi ile nihai politika seçimi ayrı hücrelerdedir.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


In [ ]:
import pandas as pd
GRID_URET = False
ANA_MODELI_SEC = False
KAYNAK_GRID_ARSIVINI_AC = False
selection = latest_run("dinamik_feature_selection")
grid = latest_run("kumeleme_karsilastirma")
GIRDI_DOSYASI = source_input(selection)
clustering_config = read_json(ROOT / "python_kodlari/ayarlar/kumeleme.json")
policy = read_json(ROOT / "python_kodlari/ayarlar/otomatik_kumeleme.json")
display(clustering_config)
display(policy)
if KAYNAK_GRID_ARSIVINI_AC:
    grid = extract_bundle("clustering_package.zip")
print("Değişken seçimi:", selection)
print("Kaynak grid:", grid)


## Grid üretimi

03’te tamamlanmış seçim gerekir. K-Prototypes için karma sayısal/kategorik alan kalmamışsa
yöntem “uygulanamaz” olarak kalır; gizlice başka yöntem kullanılmaz. Kararlılık deneyleri
tekrarlı olduğu için bu aşama uzun sürebilir.


In [ ]:
if GRID_URET:
    if selection is None:
        raise FileNotFoundError("Önce 03’te değişken seçimini çalıştırın.")
    configs = prepare_configs(GIRDI_DOSYASI, {"kumeleme.json": clustering_config})
    grid = fresh_run("kumeleme_karsilastirma")
    run_python("kumeleme_karsilastir.py", "run", "--selection", selection, "--config", configs / "kumeleme.json", "--output", grid)
    run_python("kumeleme_karsilastir.py", "check", "--run", grid)
else:
    print("Grid yeniden eğitilmedi.")


## Ortak ana model seçimi

Ekim–Kasım 2026 seçimde, Aralık 2026 ayrı tarihsel kontrolde kullanılır. Her grubun kendi
doğrulama silhouette ortalaması en az **0,10** olmalı. Banka geneli yüksek puan zayıf grubu
örtemez. DEA ana ve ek girdi senaryoları için mevcut ortak küme alt sınırı **15**’tir.
Hiçbir aday bütün koşulları sağlamazsa seçim yapılmaz; şube dışarıda bırakılmaz ve eşik
kendiliğinden düşürülmez. Bu eşikler pilot politika tercihleridir.


In [ ]:
if ANA_MODELI_SEC:
    if grid is None:
        raise FileNotFoundError("Önce grid üretin veya kaynak grid arşivini açın.")
    configs = prepare_configs(GIRDI_DOSYASI, {"otomatik_kumeleme.json": policy})
    chosen = fresh_run("otomatik_kumeleme")
    run_python("otomatik_kumeleme.py", "select", "--source", grid, "--config", configs / "otomatik_kumeleme.json", "--output", chosen)
    run_python("otomatik_kumeleme.py", "verify", "--run", chosen)
else:
    print("Yeni politika seçimi yapılmadı.")


In [ ]:
scores = read_table(result_path("otomatik_kumeleme", "candidate_scores.csv", "auto_cluster_scores.csv"))
display(scores[["candidate_id", "applicable", "qualified", "score", "worst_group_silhouette", "failed_gates"]])
decision = current_decision()
display(decision)
print("Onaylanan ortak aday:", decision.get("selected"))


In [ ]:
geo = read_table(result_path("otomatik_kumeleme", "geographic_quality.csv", "auto_cluster_geographic_quality.csv"))
display(geo)
display(read_json(result_path("otomatik_kumeleme", "full_coverage.json", "auto_cluster_full_coverage.json")))
if not decision.get("selected"):
    print("Beklenen durum: uygun ana model yok; 05 ve 06’da yeni DEA / atama başlatılmaz.")
